# 🏦 Análisis de Fraude — Transferencias Inmediatas
## Versión 1 — Base reusable para futuros análisis

**Equipo**: Prevención de Fraude y Gobierno Operacional | Scotiabank Perú

---

### 📋 Contexto del análisis

**Tipo de transacción analizada**: Transferencias inmediatas Scotiabank → bancos externos (BBVA, BCP, Interbank, GMONEY, etc.)

**NO incluye**: 
- Pagos de tarjeta de crédito
- Transferencias entre cuentas Scotiabank
- E-commerce o transacciones con tarjeta

**Hallazgo previo del EDA**: el fraude se concentra en banco destino **Interbank** tanto en monto como en cantidad.

---

### 🎯 Objetivo

Construir un framework reusable de ingeniería de variables y evaluación de reglas de control que permita:
1. Identificar patrones de fraude en transferencias inmediatas
2. Establecer umbrales y reglas con métricas claras (precisión, recall, F1)
3. Detectar relaciones sospechosas (cuentas mula, ráfagas, ingeniería social)
4. Servir de base para futuros análisis (tarjeta presente, e-commerce)

---

### 📚 Familias de features que vamos a construir

| Familia | Variables | Para qué sirve |
|---|---|---|
| **Temporales** | hora, día semana, nocturno, fin de semana | Detectar transacciones fuera de patrón habitual |
| **Velocidad** | n_trx en 3min/5min/1h/12h/24h | Detectar ráfagas (ingeniería social) |
| **Monto** | monto acumulado por ventana, ratio vs segmento | Detectar montos anómalos |
| **Destino** | banco riesgo, primera vez a esa cuenta | Detectar destinos sospechosos |
| **Red** | n clientes a una cuenta, cuentas mula | Detectar redes de fraude |
| **Dispositivo** | hash, IP, n dispositivos por cliente | Detectar suplantación |
| **Cliente** | edad, segmento | Caracterizar el perfil de víctimas |

---
## ⚙️ CELDA 1 — Configuración

Aquí defines las rutas y los nombres de las columnas. **Es la única parte del script que necesitas modificar** cuando cambien los datos.

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path

print(f'Polars version: {pl.__version__}')
print(f'Pandas version: {pd.__version__}')

# ── RUTA DEL ARCHIVO ──────────────────────────────────────────────────────────
RUTA_8750 = r'C:\Users\s4930359\OneDrive - The Bank of Nova Scotia\Seguimiento_digital\consolidado_digital.xlsx'

# ── DICCIONARIO DE COLUMNAS ─────────────────────────────────────────────────
# Mapeo: 'nombre_estandar' (para usar en el código) → 'NOMBRE_REAL_EN_EXCEL'
COLUMNAS = {
    # Identificadores
    'cliente_id':         'ACF-ID CLIENTE',
    'cuenta_origen':      'ACF-CUENTA ORIGEN',
    'cuenta_destino':     'ACF-CUENTA DESTINO',
    'hash_dispositivo':   'ACF-CODIGO HASH',

    # Transacción
    'monto':              'ACF-MONTO EN MONEDA LOCAL',
    'fecha_trx':          'ACF-FECHA TRX',
    'hora_trx':           'ACF-HORA TRX',
    'banco_destino':      'BANCO DESTINO',
    'nombre_beneficiario':'ACF-NOMBRE/LOCALIZACION COMERCIO',  # nombre_comercio en realidad guarda al beneficiario
    'ip_terminal':        'ACF-ID TERMINAL/DIRECCION IP',

    # Cliente
    'segmento':           'VAA-EVENTO DE COMPROMISO OTRA FUENTE',
    'edad_cliente':       'ACF-EDAD CLIENTE',

    # Variable objetivo
    'indicador_fraude':   'ACF-INDICADOR DE FRAUDE',

    # ── PENDIENTES — descomenta cuando estén disponibles ──────────────────
    # 'saldo_disponible':  'NOMBRE_DE_LA_COLUMNA_DE_SALDO',
    # 'canal':             'NOMBRE_DE_LA_COLUMNA_DE_CANAL',
    # 'metodo_auth':       'NOMBRE_DE_LA_COLUMNA_DE_AUTH',
}

# ── PARÁMETROS DEL ANÁLISIS ──────────────────────────────────────────────────
BANCO_RIESGO    = 'INTERBANK'   # Banco destino con mayor concentración de fraude
EDAD_VULNERABLE = 60            # Umbral de edad para flag de cliente vulnerable
UMBRALES_MONTO  = [500, 1000, 1500, 2000, 3000, 5000, 7500, 10000, 15000, 20000]

OUTPUT_DIR = Path(r'C:\Users\s4930359\Data_Herramientas\data\Gold\transferencias')

print('\n✓ Configuración lista')

---
## 📥 CELDA 2 — Carga del archivo

Leemos el Excel con pandas (más robusto para Excel) y luego convertimos a Polars (más rápido para procesamiento).

**¿Por qué este flujo híbrido?** 
- pandas: mejor lector de Excel (`openpyxl`)
- Polars: rendimiento 10-50x superior en agregaciones y rolling

In [ ]:
df_pd = pd.read_excel(RUTA_8750)
print(f'Filas leídas: {len(df_pd):,}')
print(f'Columnas en el archivo: {len(df_pd.columns)}')

# Convertir a Polars
df = pl.from_pandas(df_pd)
del df_pd  # liberar memoria

# Renombrar — solo las que existen en el archivo
rename_map = {real: std for std, real in COLUMNAS.items() if real in df.columns}
df = df.rename(rename_map)

# Validar columnas faltantes
faltantes = [std for std, real in COLUMNAS.items() if real not in df.columns]
if faltantes:
    print(f'\n⚠️  Columnas configuradas pero no encontradas en el archivo: {faltantes}')

print(f'\n✓ {df.shape[0]:,} filas | {df.shape[1]} columnas listas')
df.head(5)

---
## 🔧 CELDA 3 — Preparación de tipos

Pasos críticos:
1. Combinar fecha + hora en un único `datetime`
2. Convertir monto y edad a numérico
3. Crear variable objetivo binaria `es_fraude` (F=1, resto=0)
4. **Eliminar filas con datetime nulo** (esto era lo que causaba el error de rolling)
5. Ordenar por cliente y tiempo

In [ ]:
# ── Parsear datetime combinando fecha y hora ─────────────────────────────────
if 'fecha_trx' in df.columns and 'hora_trx' in df.columns:
    df = df.with_columns(
        pl.concat_str([
            pl.col('fecha_trx').cast(pl.Utf8),
            pl.col('hora_trx').cast(pl.Utf8)
        ], separator=' ')
          .str.to_datetime(strict=False)
          .alias('datetime_trx')
    )

# ── Tipos numéricos ───────────────────────────────────────────────────────────
df = df.with_columns([
    pl.col('monto').cast(pl.Float64, strict=False),
])
if 'edad_cliente' in df.columns:
    df = df.with_columns(pl.col('edad_cliente').cast(pl.Int32, strict=False))

# ── Variable objetivo: F=Fraude → 1, resto → 0 ──────────────────────────────
df = df.with_columns(
    (pl.col('indicador_fraude') == 'F').cast(pl.Int8).alias('es_fraude')
)

# ── CRÍTICO: Eliminar filas con datetime nulo ────────────────────────────────
# rolling_sum_by exige que el índice temporal NO tenga nulos
n_antes = df.shape[0]
df = df.filter(pl.col('datetime_trx').is_not_null())
n_despues = df.shape[0]
if n_antes != n_despues:
    print(f'⚠️  {n_antes - n_despues:,} filas eliminadas por datetime inválido')

# ── Ordenar — REQUERIDO para rolling ─────────────────────────────────────────
df = df.sort(['cliente_id', 'datetime_trx'])

# ── Estadísticas globales ─────────────────────────────────────────────────────
total_fraudes      = df['es_fraude'].sum()
total_trx          = df.shape[0]
monto_fraude_total = df.filter(pl.col('es_fraude') == 1)['monto'].sum()

print(f'Filas finales      : {total_trx:,}')
print(f'Fraudes (F)        : {total_fraudes:,}')
print(f'Fraud rate global  : {total_fraudes/total_trx*100:.3f}%')
print(f'Monto total fraude : S/ {monto_fraude_total:,.0f}')

---
## 📊 CELDA 4 — Distribución por indicador de fraude

Antes de construir features, necesitamos entender la composición de la data:

| Código | Significado |
|---|---|
| **F** | Fraude confirmado |
| **N** | Normal |
| **G** | Good (cliente confirmó la transacción) |
| **D** | Descarte del analista |
| **P** | Pendiente |

### 🧠 Cómo interpretar:

- Si el % de fraudes (F) es muy bajo (< 0.1%), tienes un problema de **clases desbalanceadas** — esto afectará el modelado posterior
- El monto promedio de fraude vs normal te dice si los estafadores van a montos altos o pequeños
- Un alto % en "Pendiente" indica que la data aún no está completamente clasificada

In [ ]:
resumen_ind = (
    df.group_by('indicador_fraude')
      .agg([
          pl.len().alias('n_trx'),
          pl.col('cliente_id').n_unique().alias('n_clientes'),
          pl.col('monto').sum().alias('monto_total'),
          pl.col('monto').mean().round(2).alias('monto_promedio'),
          pl.col('monto').median().round(2).alias('monto_mediana'),
          pl.col('monto').max().alias('monto_maximo'),
      ])
      .with_columns([
          (pl.col('n_trx') / pl.col('n_trx').sum() * 100).round(2).alias('pct_trx'),
          (pl.col('monto_total') / pl.col('monto_total').sum() * 100).round(2).alias('pct_monto'),
      ])
      .sort('monto_total', descending=True)
)
resumen_ind

---
## 🕐 CELDA 5 — Features temporales

### Variables creadas:
- `hora` (0-23)
- `dia_semana` (0=Lunes, 6=Domingo)
- `flag_nocturno` (1 si hora ≥ 22 o ≤ 6)
- `flag_fin_semana` (1 si sábado o domingo)
- `hora_sin`, `hora_cos` — encoding cíclico para que la hora 23 y la 00 sean cercanas, no extremos opuestos

### 🧠 Cómo interpretar el resultado:

Si la tabla muestra que `flag_nocturno=1` tiene un fraud_rate de 0.5% y `flag_nocturno=0` tiene 0.1%, eso significa que **el fraude ocurre 5 veces más en horario nocturno**. Esa es una señal fuerte para construir reglas.

**Patrón clásico de fraude por ingeniería social**: el estafador llama a la víctima fuera del horario bancario (cuando no puede llamar al banco a confirmar) y la presiona para transferir.

In [ ]:
df = df.with_columns([
    pl.col('datetime_trx').dt.hour().alias('hora'),
    pl.col('datetime_trx').dt.weekday().alias('dia_semana'),
])

df = df.with_columns([
    ((pl.col('hora') >= 22) | (pl.col('hora') <= 6))
        .cast(pl.Int8).alias('flag_nocturno'),
    (pl.col('dia_semana') >= 5)
        .cast(pl.Int8).alias('flag_fin_semana'),
    (2 * np.pi * pl.col('hora') / 24).sin().alias('hora_sin'),
    (2 * np.pi * pl.col('hora') / 24).cos().alias('hora_cos'),
])

# ── Validar discriminación ──
print('Distribución del fraude por horario nocturno:\n')
print(df.group_by('flag_nocturno').agg([
    pl.len().alias('n_trx'),
    pl.col('es_fraude').sum().alias('n_fraudes'),
    (pl.col('es_fraude').mean() * 100).round(3).alias('fraud_rate_%'),
]).sort('flag_nocturno'))

print('\nDistribución del fraude por día de la semana (0=Lun ... 6=Dom):\n')
print(df.group_by('dia_semana').agg([
    pl.len().alias('n_trx'),
    pl.col('es_fraude').sum().alias('n_fraudes'),
    (pl.col('es_fraude').mean() * 100).round(3).alias('fraud_rate_%'),
]).sort('dia_semana'))

---
## ⚡ CELDA 6 — Tiempo desde última transacción

Variable: `seg_desde_ultima_trx` — segundos transcurridos desde la transferencia anterior del **mismo cliente**.

### 🧠 Cómo interpretar:

- Cliente normal: días o semanas entre transferencias (mediana esperada > 86,400 seg = 1 día)
- Cliente bajo fraude por ingeniería social: minutos entre transferencias (mediana < 600 seg = 10 min)

**Si la mediana de fraude es mucho menor que la de normal**, confirmamos que el patrón de ráfaga existe.

In [ ]:
df = df.with_columns(
    pl.col('datetime_trx')
      .diff()
      .over('cliente_id')
      .dt.total_seconds()
      .alias('seg_desde_ultima_trx')
)

print('Tiempo desde última transacción del cliente — Fraude vs Normal:\n')
print(
    df.filter(pl.col('seg_desde_ultima_trx').is_not_null())
      .group_by('es_fraude')
      .agg([
          pl.len().alias('n_trx'),
          pl.col('seg_desde_ultima_trx').mean().round(0).alias('promedio_seg'),
          pl.col('seg_desde_ultima_trx').median().round(0).alias('mediana_seg'),
          pl.col('seg_desde_ultima_trx').quantile(0.10).round(0).alias('p10_seg'),
          pl.col('seg_desde_ultima_trx').quantile(0.25).round(0).alias('p25_seg'),
      ])
      .sort('es_fraude')
)
print('\nReferencia: 60 seg = 1 min | 3,600 seg = 1 h | 86,400 seg = 1 día')

---
## 🚀 CELDA 7 — Ventanas de velocidad rolling (el corazón del análisis)

### ¿Qué hacemos?

Para cada transacción de cada cliente, miramos hacia atrás en el tiempo y contamos:
- **n_trx_3min**: ¿Cuántas transferencias hizo este cliente en los últimos 3 minutos?
- **n_trx_5min**: ¿Cuántas en los últimos 5 minutos?
- **n_trx_1h, 12h, 24h**: similar para esas ventanas
- **monto_acum_X**: monto total transferido en esa ventana

### 🧠 Cómo interpretar:

Pensemos en un cliente normal:
- Hace 1-2 transferencias por semana
- `n_trx_5min = 1` casi siempre (solo la transacción actual)
- `n_trx_24h = 1 o 2` máximo

Cliente bajo fraude por ingeniería social:
- El estafador lo presiona para hacer 3-5 transferencias rápidas
- `n_trx_5min = 3, 4, 5` → MUY anómalo
- `monto_acum_5min = S/15,000` cuando normalmente transfiere S/500 → MUY anómalo

### 📐 Sintaxis Polars 1.x correcta

Usamos `df.rolling(index_column=..., period=..., group_by=...)` que es la API oficial documentada para rolling temporal por grupos.

In [ ]:
ventanas = {
    '3min': '3m',
    '5min': '5m',
    '1h':   '1h',
    '12h':  '12h',
    '24h':  '24h',
}

# Asegurar orden y ausencia de nulos
df = df.sort(['cliente_id', 'datetime_trx'])
assert df['datetime_trx'].is_null().sum() == 0, 'Hay nulos en datetime_trx'

# Marcamos sorted para evitar el escaneo de validación interno (más rápido)
df = df.set_sorted('datetime_trx')

for nombre, periodo in ventanas.items():
    print(f'  Calculando ventana {nombre}...')
    agregado = (
        df.rolling(index_column='datetime_trx', period=periodo, group_by='cliente_id')
          .agg([
              pl.col('monto').sum().alias(f'monto_acum_{nombre}'),
              pl.len().cast(pl.Int32).alias(f'n_trx_{nombre}'),
          ])
    )
    df = df.join(agregado, on=['cliente_id', 'datetime_trx'], how='left')

print('\n✓ Ventanas rolling construidas')

# ── Validar resultados ────────────────────────────────────────────────────────
print('\nComparación n_trx_5min — Fraude vs Normal:')
print(
    df.group_by('es_fraude')
      .agg([
          pl.col('n_trx_5min').mean().round(2).alias('n_trx_5min_prom'),
          pl.col('n_trx_5min').max().alias('n_trx_5min_max'),
          pl.col('monto_acum_5min').mean().round(0).alias('monto_acum_5min_prom'),
          pl.col('n_trx_24h').mean().round(2).alias('n_trx_24h_prom'),
          pl.col('n_trx_24h').max().alias('n_trx_24h_max'),
      ])
      .sort('es_fraude')
)
print('\nLectura: si n_trx_5min_prom es notablemente mayor en fraude (es_fraude=1),')
print('confirmamos que el patrón de ráfaga existe en la data.')

---
## 🏦 CELDA 8 — Banco destino y fraud rate por banco

### Variables:
- `flag_dest_riesgo` (1 si banco destino es Interbank)
- Tabla con fraud rate por banco destino

### 🧠 Cómo interpretar:

Si Interbank tiene fraud_rate de 0.8% mientras BCP tiene 0.05%, eso significa que las transferencias hacia Interbank son **16 veces más propensas a ser fraude** que hacia BCP.

**Pregunta para investigar con el especialista**: ¿Por qué Interbank? Posibles hipótesis:
1. Interbank tiene menor fricción para abrir cuentas → estafadores prefieren ese banco para crear cuentas mula
2. Procesos de KYC más débiles
3. Sesgo en la captura de la data (revisar)

In [ ]:
df = df.with_columns(
    pl.col('banco_destino')
      .str.to_uppercase()
      .str.contains(BANCO_RIESGO)
      .cast(pl.Int8)
      .alias('flag_dest_riesgo')
)

print('Fraud rate por banco destino (Top 10 por monto de fraude):\n')
print(
    df.group_by('banco_destino')
      .agg([
          pl.len().alias('n_trx'),
          pl.col('es_fraude').sum().alias('n_fraudes'),
          (pl.col('es_fraude').mean() * 100).round(3).alias('fraud_rate_%'),
          pl.col('monto').filter(pl.col('es_fraude') == 1).sum().alias('monto_fraude_S/'),
      ])
      .sort('monto_fraude_S/', descending=True)
      .head(10)
)

---
## 🆕 CELDA 9 — Primera vez transfiriendo a esa cuenta

Variable: `es_primera_vez_destino` (1 si es la primera vez que ese cliente transfiere a esa cuenta destino dentro de la data)

### 🧠 Cómo interpretar:

Una víctima de fraude por ingeniería social transfiere a una cuenta **que nunca había usado antes** (la cuenta del estafador). Si la fraud rate de `es_primera_vez_destino=1` es notablemente mayor que la de `=0`, confirmamos esta hipótesis.

**Limitación conocida**: como solo tenemos 1 mes de data, no sabemos si la cuenta destino era "conocida" antes de marzo. Esta variable es una aproximación dentro del periodo analizado.

In [ ]:
df = df.with_columns(
    pl.col('datetime_trx')
      .rank('ordinal')
      .over(['cliente_id', 'cuenta_destino'])
      .alias('_rank_cuenta')
)
df = df.with_columns(
    (pl.col('_rank_cuenta') == 1).cast(pl.Int8).alias('es_primera_vez_destino')
).drop('_rank_cuenta')

print('Fraud rate: Primera vez vs cuenta conocida:\n')
print(
    df.group_by('es_primera_vez_destino')
      .agg([
          pl.len().alias('n_trx'),
          pl.col('es_fraude').sum().alias('n_fraudes'),
          (pl.col('es_fraude').mean() * 100).round(3).alias('fraud_rate_%'),
      ])
      .sort('es_primera_vez_destino')
)

---
## 🕸️ CELDA 10 — Análisis de relación cuenta-a-cuenta (cuentas mula)

### El análisis específico que pediste

**Hipótesis**: si varios clientes de Scotiabank transfieren a la **misma cuenta destino** del mismo banco externo, esa cuenta destino es muy probablemente una cuenta mula.

### Variables:
- `n_clientes_a_cuenta_destino` — clientes Scotia distintos que enviaron a esa cuenta
- `n_trx_a_cuenta_destino` — total de trx recibidas
- `monto_total_recibido_cuenta` — monto acumulado
- `n_clientes_a_cuenta_destino_riesgo` — específicamente para banco riesgo

### 🧠 Cómo interpretar:

Una cuenta destino normal (de un familiar, comercio, etc.) recibe de **1, máximo 2 clientes Scotia distintos** en un mes. Si una cuenta recibe de 5, 10, 20 clientes distintos, es altísimamente sospechosa de ser cuenta mula.

**Caso particular Interbank**: como ya identificamos que Interbank concentra el fraude, vamos a ver específicamente qué cuentas de Interbank reciben de múltiples clientes Scotia.

In [ ]:
# ── Métricas a nivel de cuenta destino ───────────────────────────────────────
receptores = (
    df.group_by('cuenta_destino')
      .agg([
          pl.col('cliente_id').n_unique().alias('n_clientes_a_cuenta_destino'),
          pl.len().alias('n_trx_a_cuenta_destino'),
          pl.col('monto').sum().alias('monto_total_recibido_cuenta'),
      ])
)
df = df.join(receptores, on='cuenta_destino', how='left')

# ── Distribución: ¿cuántas cuentas reciben de muchos clientes? ────────────────
print('Distribución de cuentas destino por # clientes Scotia distintos:\n')
dist = (
    df.group_by('n_clientes_a_cuenta_destino')
      .agg([
          pl.col('cuenta_destino').n_unique().alias('n_cuentas_destino'),
          pl.col('es_fraude').sum().alias('n_fraudes_asociados'),
          (pl.col('es_fraude').mean() * 100).round(3).alias('fraud_rate_%'),
      ])
      .sort('n_clientes_a_cuenta_destino', descending=True)
)
print(dist.head(15))

# ── Top 20 cuentas mula sospechosas — TODOS los bancos ────────────────────────
print('\n\nTOP 20 cuentas mula sospechosas (todos los bancos):\n')
top_mula = (
    df.group_by(['cuenta_destino', 'banco_destino'])
      .agg([
          pl.col('cliente_id').n_unique().alias('n_clientes_distintos'),
          pl.len().alias('n_trx_recibidas'),
          pl.col('monto').sum().alias('monto_total_recibido'),
          pl.col('es_fraude').sum().alias('n_fraudes_asociados'),
          (pl.col('es_fraude').mean() * 100).round(3).alias('fraud_rate_%'),
      ])
      .filter(pl.col('n_clientes_distintos') >= 2)
      .sort('n_clientes_distintos', descending=True)
      .head(20)
)
print(top_mula)

# ── Top 20 cuentas mula sospechosas — SOLO BANCO RIESGO ──────────────────────
print(f'\n\nTOP 20 cuentas mula sospechosas — SOLO {BANCO_RIESGO}:\n')
top_mula_riesgo = (
    df.filter(pl.col('flag_dest_riesgo') == 1)
      .group_by(['cuenta_destino', 'banco_destino'])
      .agg([
          pl.col('cliente_id').n_unique().alias('n_clientes_distintos'),
          pl.len().alias('n_trx_recibidas'),
          pl.col('monto').sum().alias('monto_total_recibido'),
          pl.col('es_fraude').sum().alias('n_fraudes_asociados'),
          (pl.col('es_fraude').mean() * 100).round(3).alias('fraud_rate_%'),
      ])
      .filter(pl.col('n_clientes_distintos') >= 2)
      .sort('n_clientes_distintos', descending=True)
      .head(20)
)
print(top_mula_riesgo)

---
## 📱 CELDA 11 — Análisis de dispositivo (hash) e IP

### Variables:
- `n_dispositivos_cliente` — cuántos hash distintos usó el cliente en el mes
- `n_clientes_por_dispositivo` — cuántos clientes distintos usaron el mismo hash
- `n_ips_cliente` — cuántas IPs distintas usó el cliente
- `n_clientes_por_ip` — cuántos clientes distintos compartieron una IP
- `flag_dispositivo_compartido` (1 si > 1 cliente usó ese dispositivo)
- `flag_cliente_multidispositivo` (1 si el cliente usó muchos dispositivos)

### 🧠 Cómo interpretar:

**Dispositivo compartido (n_clientes_por_dispositivo > 1)**:
Patrón clásico de fraude organizado: el estafador instala la app en su propio celular y va logueándose con credenciales robadas de varios clientes.

**Cliente multidispositivo (n_dispositivos_cliente > 3 en un mes)**:
Cliente normal: 1, a veces 2 dispositivos. Suplantación: muchos dispositivos en poco tiempo.

**IP compartida**:
Similar al hash pero a nivel de red. Puede ser falso positivo (familias, oficinas) — combinarlo con hash es más preciso.

In [ ]:
# ── Métricas a nivel de hash de dispositivo ──────────────────────────────────
if 'hash_dispositivo' in df.columns:
    hash_stats = (
        df.filter(pl.col('hash_dispositivo').is_not_null())
          .group_by('hash_dispositivo')
          .agg([
              pl.col('cliente_id').n_unique().alias('n_clientes_por_dispositivo'),
              pl.len().alias('n_trx_dispositivo'),
          ])
    )
    df = df.join(hash_stats, on='hash_dispositivo', how='left')
    df = df.with_columns(
        (pl.col('n_clientes_por_dispositivo') > 1)
            .cast(pl.Int8).alias('flag_dispositivo_compartido')
    )

    # Métricas a nivel de cliente
    cliente_disp = (
        df.filter(pl.col('hash_dispositivo').is_not_null())
          .group_by('cliente_id')
          .agg([pl.col('hash_dispositivo').n_unique().alias('n_dispositivos_cliente')])
    )
    df = df.join(cliente_disp, on='cliente_id', how='left')
    df = df.with_columns(
        (pl.col('n_dispositivos_cliente') >= 3)
            .cast(pl.Int8).alias('flag_cliente_multidispositivo')
    )

    print('Fraud rate por dispositivo compartido:\n')
    print(df.group_by('flag_dispositivo_compartido').agg([
        pl.len().alias('n_trx'),
        pl.col('es_fraude').sum().alias('n_fraudes'),
        (pl.col('es_fraude').mean() * 100).round(3).alias('fraud_rate_%'),
    ]).sort('flag_dispositivo_compartido'))

    print('\nFraud rate por cliente multidispositivo (>= 3 hash distintos):\n')
    print(df.group_by('flag_cliente_multidispositivo').agg([
        pl.len().alias('n_trx'),
        pl.col('es_fraude').sum().alias('n_fraudes'),
        (pl.col('es_fraude').mean() * 100).round(3).alias('fraud_rate_%'),
    ]).sort('flag_cliente_multidispositivo'))

# ── Métricas a nivel de IP ───────────────────────────────────────────────────
if 'ip_terminal' in df.columns:
    ip_stats = (
        df.filter(pl.col('ip_terminal').is_not_null())
          .group_by('ip_terminal')
          .agg([pl.col('cliente_id').n_unique().alias('n_clientes_por_ip')])
    )
    df = df.join(ip_stats, on='ip_terminal', how='left')
    df = df.with_columns(
        (pl.col('n_clientes_por_ip') > 1).cast(pl.Int8).alias('flag_ip_compartida')
    )

    print('\nFraud rate por IP compartida:\n')
    print(df.group_by('flag_ip_compartida').agg([
        pl.len().alias('n_trx'),
        pl.col('es_fraude').sum().alias('n_fraudes'),
        (pl.col('es_fraude').mean() * 100).round(3).alias('fraud_rate_%'),
    ]).sort('flag_ip_compartida'))

---
## 👤 CELDA 12 — Edad del cliente y segmento

### Variables:
- `flag_edad_vulnerable` (1 si edad ≥ 60) — adultos mayores son víctimas frecuentes de ingeniería social
- `ratio_monto_vs_segmento` — qué tanto se desvía el monto del promedio del segmento

### 🧠 Cómo interpretar:

**Edad vulnerable**: si los adultos mayores tienen fraud_rate notablemente mayor, es un patrón claro y sirve para reglas de control adicionales (ej. doble verificación para mayores de 60)

**Ratio monto vs segmento**: un cliente del segmento masivo transfiriendo S/15,000 cuando el promedio del segmento es S/300 → ratio = 50, altamente sospechoso

In [ ]:
# ── Edad ─────────────────────────────────────────────────────────────────────
if 'edad_cliente' in df.columns:
    df = df.with_columns(
        (pl.col('edad_cliente') >= EDAD_VULNERABLE)
            .cast(pl.Int8).alias('flag_edad_vulnerable')
    )

    print(f'Fraud rate por edad vulnerable (>= {EDAD_VULNERABLE} años):\n')
    print(df.filter(pl.col('edad_cliente').is_not_null())
            .group_by('flag_edad_vulnerable').agg([
        pl.len().alias('n_trx'),
        pl.col('es_fraude').sum().alias('n_fraudes'),
        (pl.col('es_fraude').mean() * 100).round(3).alias('fraud_rate_%'),
    ]).sort('flag_edad_vulnerable'))

    # ── Análisis por rangos de edad ──
    print('\nFraud rate por rangos de edad:\n')
    df_edad = df.filter(pl.col('edad_cliente').is_not_null()).with_columns(
        pl.when(pl.col('edad_cliente') < 25).then(pl.lit('1. <25'))
          .when(pl.col('edad_cliente') < 35).then(pl.lit('2. 25-34'))
          .when(pl.col('edad_cliente') < 50).then(pl.lit('3. 35-49'))
          .when(pl.col('edad_cliente') < 60).then(pl.lit('4. 50-59'))
          .otherwise(pl.lit('5. 60+'))
          .alias('rango_edad')
    )
    print(df_edad.group_by('rango_edad').agg([
        pl.len().alias('n_trx'),
        pl.col('es_fraude').sum().alias('n_fraudes'),
        (pl.col('es_fraude').mean() * 100).round(3).alias('fraud_rate_%'),
    ]).sort('rango_edad'))

# ── Ratio monto vs segmento ──────────────────────────────────────────────────
if 'segmento' in df.columns:
    avg_seg = df.group_by('segmento').agg(
        pl.col('monto').mean().alias('avg_monto_segmento')
    )
    df = df.join(avg_seg, on='segmento', how='left')
    df = df.with_columns(
        (pl.col('monto') / (pl.col('avg_monto_segmento') + 1))
            .alias('ratio_monto_vs_segmento')
    )

    print('\nRatio monto vs segmento — Fraude vs Normal:\n')
    print(df.group_by('es_fraude').agg([
        pl.col('ratio_monto_vs_segmento').mean().round(2).alias('ratio_prom'),
        pl.col('ratio_monto_vs_segmento').median().round(2).alias('ratio_mediana'),
        pl.col('ratio_monto_vs_segmento').max().round(2).alias('ratio_maximo'),
    ]).sort('es_fraude'))

---
## 🔗 CELDA 13 — Features compuestas

Combinan varias señales débiles en una sola señal fuerte.

### Reglas compuestas:
- `flag_rafaga_dest_riesgo` — ráfaga 5min + banco riesgo
- `flag_alto_riesgo` — ráfaga + nocturno + banco riesgo
- `flag_mula_destino` — primera vez + cuenta recibe de varios clientes
- `flag_dispositivo_compartido_riesgo` — hash compartido + banco riesgo

### 🧠 Cómo interpretar:

Una señal débil sola puede dar muchos falsos positivos. Pero **3 señales débiles que coinciden** en la misma transacción ya es altamente sospechoso. El truco está en encontrar la combinación que maximiza precisión sin sacrificar mucho recall.

In [ ]:
df = df.with_columns([
    # Ráfaga hacia banco de riesgo
    ((pl.col('n_trx_5min') >= 2) & (pl.col('flag_dest_riesgo') == 1))
        .cast(pl.Int8).alias('flag_rafaga_dest_riesgo'),
    # Triple señal
    ((pl.col('n_trx_5min') >= 2) &
     (pl.col('flag_nocturno') == 1) &
     (pl.col('flag_dest_riesgo') == 1))
        .cast(pl.Int8).alias('flag_alto_riesgo'),
    # Primera vez + cuenta recibe de varios
    ((pl.col('es_primera_vez_destino') == 1) &
     (pl.col('n_clientes_a_cuenta_destino') >= 2))
        .cast(pl.Int8).alias('flag_mula_destino'),
])

# Si tenemos hash, agregamos otra compuesta
if 'flag_dispositivo_compartido' in df.columns:
    df = df.with_columns(
        ((pl.col('flag_dispositivo_compartido') == 1) &
         (pl.col('flag_dest_riesgo') == 1))
            .cast(pl.Int8).alias('flag_dispositivo_compartido_riesgo')
    )

print(f'✓ Features compuestas creadas')
print(f'Total columnas en el dataset: {df.shape[1]}')

features_creadas = [c for c in df.columns if c.startswith(('flag_', 'n_trx_', 'monto_acum_',
                                                            'es_primera', 'seg_desde',
                                                            'ratio_', 'n_clientes_', 'n_dispositivos',
                                                            'n_ips', 'rango_'))]
print(f'\nFeatures construidas ({len(features_creadas)}):')
for f in features_creadas:
    print(f'  - {f}')

---
## 📈 CELDA 14 — Análisis de discriminación (Lift)

### Métricas de cada feature binaria:
- **fraud_rate_flag1**: % de fraude cuando la condición se activa
- **fraud_rate_flag0**: % de fraude cuando NO se activa
- **lift**: cociente entre ambas — mientras más alto, mejor discrimina
- **fraudes_capturados**: # de fraudes que la feature alerta
- **monto_fraude_cap**: S/ de fraude que la feature alerta

### 🧠 Cómo interpretar el lift:

| Lift | Significado |
|---|---|
| 1 | La feature no discrimina |
| 2 | 2x más probable de fraude |
| 5 | 5x más probable — feature útil |
| 10+ | Excelente discriminador |

**Las features con mayor lift serán los componentes principales de tus reglas de control**.

In [ ]:
flags_a_evaluar = [
    'flag_nocturno',
    'flag_fin_semana',
    'flag_dest_riesgo',
    'es_primera_vez_destino',
    'flag_rafaga_dest_riesgo',
    'flag_alto_riesgo',
    'flag_mula_destino',
    'flag_dispositivo_compartido',
    'flag_cliente_multidispositivo',
    'flag_ip_compartida',
    'flag_dispositivo_compartido_riesgo',
    'flag_edad_vulnerable',
]

filas_disc = []
for flag in flags_a_evaluar:
    if flag not in df.columns:
        continue
    t = df.group_by(flag).agg([
        pl.len().alias('n_trx'),
        pl.col('es_fraude').sum().alias('n_fraudes'),
        pl.col('es_fraude').mean().alias('fraud_rate'),
        pl.col('monto').filter(pl.col('es_fraude') == 1).sum().alias('monto_fraude'),
    ])
    f1 = t.filter(pl.col(flag) == 1)
    f0 = t.filter(pl.col(flag) == 0)
    if f1.is_empty() or f0.is_empty():
        continue
    fr1, fr0 = f1['fraud_rate'][0], f0['fraud_rate'][0]
    filas_disc.append({
        'feature':            flag,
        'fr_flag1_%':         round(fr1 * 100, 3),
        'fr_flag0_%':         round(fr0 * 100, 3),
        'lift':               round(fr1 / (fr0 + 1e-10), 2),
        'n_trx_alertadas':    f1['n_trx'][0],
        'fraudes_capturados': f1['n_fraudes'][0],
        'monto_fraude_S/':    round(f1['monto_fraude'][0] or 0, 0),
    })

disc_df = pl.DataFrame(filas_disc).sort('lift', descending=True)
print('Features ordenadas por poder discriminante (lift):\n')
disc_df

---
## 💰 CELDA 15 — Tabla de umbrales de monto

**Pregunta que responde**: si pongo el control a partir de un monto X, ¿cuánto fraude capturo y a cuántos clientes normales afecto?

### 🧠 Cómo interpretar:

El balance ideal es maximizar **recall** (capturar la mayor cantidad de fraudes) minimizando el **% normales afectados**. 

Por ejemplo, si el umbral S/3,000 te da:
- recall_% = 75 (capturas 75% del fraude)
- pct_normales_afect_% = 8 (solo afectas al 8% de clientes normales)

Eso es un buen punto de control — capturas la mayoría del fraude con poca fricción al cliente.

Comparamos dos versiones:
1. **Solo monto**: amplio, captura más fraude pero más fricción
2. **Monto + banco riesgo**: quirúrgico, menos fricción

In [ ]:
total_normales = df.shape[0] - total_fraudes

# ── Tabla 1: Solo por monto ───────────────────────────────────────────────────
filas_u = []
for u in UMBRALES_MONTO:
    sub = df.filter(pl.col('monto') >= u)
    nf  = sub['es_fraude'].sum()
    nn  = sub.shape[0] - nf
    mc  = sub.filter(pl.col('es_fraude') == 1)['monto'].sum()
    filas_u.append({
        'umbral_S/':            u,
        'fraudes_captados':     int(nf),
        'recall_%':             round(nf / (total_fraudes + 1e-10) * 100, 1),
        'monto_cap_S/':         round(mc, 0),
        'pct_monto_fraude_%':   round(mc / monto_fraude_total * 100, 1),
        'normales_afectados':   int(nn),
        'pct_normales_%':       round(nn / (total_normales + 1e-10) * 100, 1),
        'precision_%':          round(nf / (sub.shape[0] + 1e-10) * 100, 2),
    })

tabla_u = pl.DataFrame(filas_u)
print('TABLA 1 — Control por monto solo:\n')
print(tabla_u)

# ── Tabla 2: Monto + banco riesgo ─────────────────────────────────────────────
filas_uc = []
for u in UMBRALES_MONTO:
    sub = df.filter((pl.col('monto') >= u) & (pl.col('flag_dest_riesgo') == 1))
    if sub.shape[0] == 0:
        continue
    nf = sub['es_fraude'].sum()
    nn = sub.shape[0] - nf
    mc = sub.filter(pl.col('es_fraude') == 1)['monto'].sum()
    filas_uc.append({
        'umbral_S/':            u,
        'fraudes_captados':     int(nf),
        'recall_%':             round(nf / (total_fraudes + 1e-10) * 100, 1),
        'monto_cap_S/':         round(mc, 0),
        'normales_afectados':   int(nn),
        'precision_%':          round(nf / (sub.shape[0] + 1e-10) * 100, 2),
    })

tabla_uc = pl.DataFrame(filas_uc)
print(f'\nTABLA 2 — Control por monto + banco {BANCO_RIESGO}:\n')
print(tabla_uc)

---
## 🎯 CELDA 16 — Evaluación de reglas de control

Probamos varias combinaciones de reglas y medimos:
- **Recall**: % de fraudes capturados
- **Precision**: % de alertas que son fraude real
- **F1**: balance recall/precision
- **Monto fraude capturado** en soles (impacto económico real)

### 🧠 Cómo elegir la mejor regla:

No siempre la de mayor F1 es la mejor. Considera:
- **Costo de falso positivo**: cliente bloqueado se molesta, llama, deserta
- **Costo de falso negativo**: pérdida económica directa por fraude

En banca, el costo del fraude suele ser mucho mayor que la fricción al cliente bueno, por lo que se prioriza el **recall** por encima de la precisión, aceptando algunos falsos positivos.

In [ ]:
def evaluar_regla(df, nombre, expr):
    df_e      = df.with_columns(expr.cast(pl.Int8).alias('_alerta'))
    tp        = df_e.filter((pl.col('_alerta') == 1) & (pl.col('es_fraude') == 1)).shape[0]
    fp        = df_e.filter((pl.col('_alerta') == 1) & (pl.col('es_fraude') == 0)).shape[0]
    total_f   = int(df['es_fraude'].sum())
    total_a   = tp + fp
    prec      = tp / (total_a + 1e-10)
    rec       = tp / (total_f + 1e-10)
    f1        = 2 * prec * rec / (prec + rec + 1e-10)
    monto_c   = df_e.filter((pl.col('_alerta') == 1) & (pl.col('es_fraude') == 1))['monto'].sum()
    monto_tf  = df.filter(pl.col('es_fraude') == 1)['monto'].sum()
    print(f"\n  {'─'*55}")
    print(f'  Regla    : {nombre}')
    print(f'  Alertas  : {total_a:,}  ({total_a/df.shape[0]*100:.1f}% del universo)')
    print(f'  Recall   : {rec*100:.1f}%  ({tp:,} de {total_f:,} fraudes capturados)')
    print(f'  Precision: {prec*100:.1f}%  |  F1: {f1:.3f}')
    print(f'  FP (normales afectados) : {fp:,}')
    print(f'  Monto fraude capturado  : S/ {monto_c:,.0f}  ({monto_c/monto_tf*100:.1f}% del total)')

evaluar_regla(df, 'R1 — Banco destino riesgo',
    pl.col('flag_dest_riesgo') == 1)

evaluar_regla(df, 'R2 — Ráfaga >= 2 trx en 5 min',
    pl.col('n_trx_5min') >= 2)

evaluar_regla(df, 'R3 — Banco riesgo + primera vez a esa cuenta',
    (pl.col('flag_dest_riesgo') == 1) & (pl.col('es_primera_vez_destino') == 1))

evaluar_regla(df, 'R4 — Cuenta mula (>= 2 clientes a la misma cuenta)',
    pl.col('n_clientes_a_cuenta_destino') >= 2)

evaluar_regla(df, 'R5 — Ráfaga 5min + banco riesgo',
    pl.col('flag_rafaga_dest_riesgo') == 1)

evaluar_regla(df, 'R6 — Triple señal (ráfaga + nocturno + banco riesgo)',
    pl.col('flag_alto_riesgo') == 1)

if 'flag_dispositivo_compartido' in df.columns:
    evaluar_regla(df, 'R7 — Dispositivo compartido',
        pl.col('flag_dispositivo_compartido') == 1)

evaluar_regla(df, 'R8 — Mula destino (primera vez + cuenta recibe de varios)',
    pl.col('flag_mula_destino') == 1)

evaluar_regla(df, 'R9 — OR combinado: R3 + R5 + R8',
    ((pl.col('flag_dest_riesgo') == 1) & (pl.col('es_primera_vez_destino') == 1)) |
    (pl.col('flag_rafaga_dest_riesgo') == 1) |
    (pl.col('flag_mula_destino') == 1))

---
## 💾 CELDA 17 — Exportar resultados

Guardamos el dataset enriquecido en Parquet (formato eficiente) y las tablas de análisis en CSV para revisión y para Power BI.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset enriquecido — base para futuros análisis y modelado
df.write_parquet(OUTPUT_DIR / '8750_marzo_features.parquet')

# Tablas de análisis
disc_df.write_csv(OUTPUT_DIR / 'discriminacion_features.csv')
tabla_u.write_csv(OUTPUT_DIR / 'tabla_umbrales_monto.csv')
tabla_uc.write_csv(OUTPUT_DIR / 'tabla_umbrales_combinados.csv')
top_mula.write_csv(OUTPUT_DIR / 'cuentas_mula_todos_bancos.csv')
top_mula_riesgo.write_csv(OUTPUT_DIR / 'cuentas_mula_banco_riesgo.csv')

print(f'✓ Exportado a: {OUTPUT_DIR}\n')
print('  Archivos generados:')
print('  - 8750_marzo_features.parquet     (dataset completo con features)')
print('  - discriminacion_features.csv     (lift por feature)')
print('  - tabla_umbrales_monto.csv        (análisis por umbral de monto)')
print('  - tabla_umbrales_combinados.csv   (umbral + banco riesgo)')
print('  - cuentas_mula_todos_bancos.csv   (top cuentas sospechosas)')
print('  - cuentas_mula_banco_riesgo.csv   (top cuentas mula en banco riesgo)')

---
## 📋 Resumen ejecutivo del análisis

### Lo que encontramos:
1. **Concentración geográfica/bancaria**: el fraude se concentra en transferencias a Interbank
2. **Patrón de ráfaga**: las víctimas hacen múltiples transferencias en minutos (vs días en clientes normales)
3. **Cuentas mula**: cuentas destino que reciben de múltiples clientes Scotia distintos en el mismo mes
4. **Dispositivos compartidos**: hash que se usa con varios clientes — fuerte señal de suplantación

### Reglas de control propuestas (en orden de prioridad):

**Prioridad alta — implementar primero**:
- R5: Ráfaga + banco riesgo (alta precisión, fácil de implementar)
- R8: Mula destino (alta precisión cuando se confirma con varios clientes)

**Prioridad media — tras evaluar fricción al cliente**:
- R3: Banco riesgo + primera vez (más amplia, requiere paso de verificación)
- R6: Triple señal (muy precisa, captura menos volumen)

**A futuro — cuando estén disponibles las variables**:
- Saldo disponible → ratio_monto_saldo, flag_vaciado
- Canal → diferenciación app/web/teléfono
- Método auth → reglas específicas por nivel de seguridad

### Próximos pasos:
1. Revisar lifts y elegir las 2-3 mejores reglas con el especialista
2. Validar las cuentas mula identificadas con el equipo de detección
3. Estimar fricción al cliente (cuántas alertas/día generaría cada regla)
4. Pilotear con monitoreo manual antes de bloqueo automático